<a href="https://colab.research.google.com/github/sonalisalvi/careloop-no-show-guard/blob/main/03_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import subprocess
from pathlib import Path
import pandas as pd
import numpy as np

# Repository configuration
REPO_URL = "https://github.com/sonalisalvi/careloop-no-show-guard.git"
PROJECT_DIR = Path("/content/careloop-no-show-guard")

# Clone or update the repository
if PROJECT_DIR.exists():
    subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"],
        check=True
    )
else:
    subprocess.run(
        ["git", "clone", REPO_URL, str(PROJECT_DIR)],
        check=True
    )

# Load the cleaned appointment dataset
DATA_PATH = PROJECT_DIR / "appointments_clean.csv"

appointments = pd.read_csv(DATA_PATH)

print("========== PART 3: MACHINE LEARNING ==========")
print("Dataset loaded successfully!")
print("Dataset shape:", appointments.shape)

print("\nColumn names:")
print(appointments.columns.tolist())

print("\nFirst five records:")
display(appointments.head())

print("\nTarget variable distribution:")
print(appointments["no_show"].value_counts().sort_index())

print("\nTarget variable percentages:")
print(
    (appointments["no_show"].value_counts(normalize=True).sort_index() * 100)
    .round(1)
)


========== PART 3: MACHINE LEARNING ==========
Dataset loaded successfully!
Dataset shape: (990, 16)

Column names:
['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status', 'lead_days', 'no_show']

First five records:


,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
0,A0001,P509,C05,2025-01-24,2025-01-25,60,F,18.0,850,Phone,Consultation,Yes,1,No-Show,1,1
1,A0002,P046,C07,2025-05-27,2025-06-06,39,M,12.5,350,Walk-In,Follow-up,No,2,Attended,10,0
2,A0003,P009,C02,2025-03-24,2025-03-30,63,F,4.1,850,Walk-In,Consultation,No,0,Attended,6,0
3,A0004,P482,C01,2025-04-18,2025-04-22,52,F,7.5,750,App,Consultation,Yes,0,Attended,4,0
4,A0005,P251,C01,2024-11-23,2025-01-02,64,F,19.0,700,Phone,Consultation,No,0,No-Show,40,1



Target variable distribution:
no_show
0    772
1    218
Name: count, dtype: int64

Target variable percentages:
no_show
0    78.0
1    22.0
Name: proportion, dtype: float64


In [ ]:

# ==========================================
# STEP 2: FEATURE SELECTION AND PREPARATION
# ==========================================

# Select numerical features
numeric_features = [
    "age",
    "distance_km",
    "fee_inr",
    "lead_days",
    "prior_no_shows"
]

# Select categorical features
categorical_features = [
    "reminder_sent",
    "booking_channel",
    "visit_type"
]

# Create the input feature DataFrame
X = appointments[numeric_features + categorical_features].copy()

# Convert reminder_sent from Yes/No to 1/0
X["reminder_sent"] = X["reminder_sent"].map({
    "Yes": 1,
    "No": 0
})

# Convert categorical variables into numerical columns
X = pd.get_dummies(
    X,
    columns=["booking_channel", "visit_type"],
    dtype=int
)

# Create the target variable
y = appointments["no_show"].copy()

# Display the prepared features and target
print("========== FEATURE PREPARATION ==========")

print("\nFeature matrix shape:", X.shape)
print("Target shape:", y.shape)

print("\nFeature columns:")
print(X.columns.tolist())

print("\nFirst five prepared records:")
display(X.head())

print("\nTarget distribution:")
print(y.value_counts().sort_index())

print("\nMissing values in features:", X.isnull().sum().sum())
print("Missing values in target:", y.isnull().sum())

print("\nTotal number of features:", X.shape[1])


========== FEATURE PREPARATION ==========

Feature matrix shape: (990, 11)
Target shape: (990,)

Feature columns:
['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']

First five prepared records:


,age,distance_km,fee_inr,lead_days,prior_no_shows,reminder_sent,booking_channel_App,booking_channel_Phone,booking_channel_Walk-In,visit_type_Consultation,visit_type_Follow-up
0,60,18.0,850,1,1,1,0,1,0,1,0
1,39,12.5,350,10,2,0,0,0,1,0,1
2,63,4.1,850,6,0,0,0,0,1,1,0
3,52,7.5,750,4,0,1,1,0,0,1,0
4,64,19.0,700,40,0,0,0,1,0,1,0



Target distribution:
no_show
0    772
1    218
Name: count, dtype: int64

Missing values in features: 0
Missing values in target: 0

Total number of features: 11


In [ ]:

# ==========================================
# STEP 3: TRAIN-TEST SPLIT
# ==========================================

from sklearn.model_selection import train_test_split

# Split features and target into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Display dataset sizes
print("========== TRAIN-TEST SPLIT ==========")

print("\nTraining features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining target:", y_train.shape)
print("Testing target:", y_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts().sort_index())

print("\nTesting target distribution:")
print(y_test.value_counts().sort_index())

print("\nTraining no-show percentage:",
      round(y_train.mean() * 100, 1), "%")

print("Testing no-show percentage:",
      round(y_test.mean() * 100, 1), "%")


========== TRAIN-TEST SPLIT ==========

Training features: (792, 11)
Testing features: (198, 11)

Training target: (792,)
Testing target: (198,)

Training target distribution:
no_show
0    623
1    169
Name: count, dtype: int64

Testing target distribution:
no_show
0    149
1     49
Name: count, dtype: int64

Training no-show percentage: 21.3 %
Testing no-show percentage: 24.7 %


In [ ]:

# ==========================================
# STEP 4: FEATURE SCALING
# ==========================================

from sklearn.preprocessing import StandardScaler

# Create the scaler
scaler = StandardScaler()

# Learn scaling parameters from training data only
X_train_scaled = scaler.fit_transform(X_train)

# Apply the same parameters to testing data
X_test_scaled = scaler.transform(X_test)

print("========== FEATURE SCALING ==========")

print("Original training shape:", X_train.shape)
print("Scaled training shape:", X_train_scaled.shape)

print("\nOriginal testing shape:", X_test.shape)
print("Scaled testing shape:", X_test_scaled.shape)

# Check the scaled training data
print("\nMean of first five scaled training features:")
print(np.round(X_train_scaled.mean(axis=0)[:5], 4))

print("\nStandard deviation of first five scaled training features:")
print(np.round(X_train_scaled.std(axis=0)[:5], 4))

print("\nFirst five scaled training records:")
display(pd.DataFrame(
    X_train_scaled,
    columns=X.columns
).head())


========== FEATURE SCALING ==========
Original training shape: (792, 11)
Scaled training shape: (792, 11)

Original testing shape: (198, 11)
Scaled testing shape: (198, 11)

Mean of first five scaled training features:
[ 0.  0. -0.  0. -0.]

Standard deviation of first five scaled training features:
[1. 1. 1. 1. 1.]

First five scaled training records:


,age,distance_km,fee_inr,lead_days,prior_no_shows,reminder_sent,booking_channel_App,booking_channel_Phone,booking_channel_Walk-In,visit_type_Consultation,visit_type_Follow-up
0,0.172384,0.391234,0.906003,0.380215,-0.667437,-1.153001,-0.787167,1.322876,-0.583182,0.842961,-0.842961
1,1.673357,1.531389,-1.807649,-1.128920,-0.667437,-1.153001,-0.787167,1.322876,-0.583182,-1.186295,1.186295
2,0.592657,-0.401917,0.906003,0.264128,-0.667437,-1.153001,-0.787167,1.322876,-0.583182,0.842961,-0.842961
3,-0.428005,-0.203629,-0.179457,-0.432396,-0.667437,-1.153001,-0.787167,-0.755929,1.714730,0.842961,-0.842961
4,0.532618,2.671544,0.363273,-0.896745,-0.667437,-1.153001,-0.787167,-0.755929,1.714730,0.842961,-0.842961


In [ ]:

# ==========================================
# STEP 5: LOGISTIC REGRESSION
# ==========================================

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

# Create the Logistic Regression model
log_model = LogisticRegression(max_iter=1000)

# Train the model using training data
log_model.fit(X_train_scaled, y_train)

# Predict classes on the test dataset
y_pred = log_model.predict(X_test_scaled)

# Predict probabilities of no-show (class 1)
y_pred_proba = log_model.predict_proba(X_test_scaled)[:, 1]

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_pred_proba)

# Display results
print("========== LOGISTIC REGRESSION RESULTS ==========")

print("\nAccuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1 Score:", round(f1, 4))
print("ROC-AUC:", round(roc_auc, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred, labels=[0, 1]))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Attended (0)", "No-Show (1)"],
    zero_division=0
))


========== LOGISTIC REGRESSION RESULTS ==========

Accuracy: 0.7879
Precision: 0.6296
Recall: 0.3469
F1 Score: 0.4474
ROC-AUC: 0.7889

Confusion Matrix:
[[139  10]
 [ 32  17]]

Classification Report:
              precision    recall  f1-score   support

Attended (0)       0.81      0.93      0.87       149
 No-Show (1)       0.63      0.35      0.45        49

    accuracy                           0.79       198
   macro avg       0.72      0.64      0.66       198
weighted avg       0.77      0.79      0.76       198



In [ ]:
# Extract Logistic Regression coefficients
coefficients = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": log_model.coef_[0]
})

# Sort by coefficient
coefficients = coefficients.sort_values(
    by="Coefficient",
    ascending=False
)

print("========== LOGISTIC REGRESSION COEFFICIENTS ==========")
display(coefficients.round(4))

# Calculate the reminder odds ratio
reminder_index = list(X_train.columns).index("reminder_sent")

reminder_coefficient = log_model.coef_[0][reminder_index]
reminder_scale = scaler.scale_[reminder_index]

# Convert the scaled coefficient to the original 0-to-1 feature change
reminder_odds_ratio = np.exp(reminder_coefficient / reminder_scale)

print("\n========== REMINDER IMPACT ==========")
print("Reminder coefficient:", round(reminder_coefficient, 4))
print("Reminder odds ratio:", round(reminder_odds_ratio, 4))
print(
    "Percentage change in odds:",
    round((reminder_odds_ratio - 1) * 100, 2),
    "%"
)

========== LOGISTIC REGRESSION COEFFICIENTS ==========


,Feature,Coefficient
4,prior_no_shows,0.6496
3,lead_days,0.6145
2,fee_inr,0.4528
1,distance_km,0.2534
7,booking_channel_Phone,0.2159
9,visit_type_Consultation,0.0157
10,visit_type_Follow-up,-0.0157
6,booking_channel_App,-0.0468
8,booking_channel_Walk-In,-0.1864
0,age,-0.2539



========== REMINDER IMPACT ==========
Reminder coefficient: -0.5807
Reminder odds ratio: 0.3094
Percentage change in odds: -69.06 %


In [ ]:
# Decision Tree classifier

from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Model 1: Full Decision Tree
tree_full = DecisionTreeClassifier(random_state=42)

tree_full.fit(X_train_scaled, y_train)

tree_full_pred = tree_full.predict(X_test_scaled)
tree_full_proba = tree_full.predict_proba(X_test_scaled)[:, 1]


# Model 2: Decision Tree with maximum depth 3
tree_depth3 = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

tree_depth3.fit(X_train_scaled, y_train)

tree_depth3_pred = tree_depth3.predict(X_test_scaled)
tree_depth3_proba = tree_depth3.predict_proba(X_test_scaled)[:, 1]


# Evaluation function
def evaluate_model(name, y_true, y_pred, y_proba):

    print(f"\n========== {name} ==========")

    print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))

    print("Precision:", round(
        precision_score(y_true, y_pred, zero_division=0), 4
    ))

    print("Recall:", round(
        recall_score(y_true, y_pred, zero_division=0), 4
    ))

    print("F1 Score:", round(
        f1_score(y_true, y_pred, zero_division=0), 4
    ))

    print("ROC-AUC:", round(
        roc_auc_score(y_true, y_proba), 4
    ))


evaluate_model(
    "FULL DECISION TREE",
    y_test,
    tree_full_pred,
    tree_full_proba
)

evaluate_model(
    "DECISION TREE - MAX DEPTH 3",
    y_test,
    tree_depth3_pred,
    tree_depth3_proba
)


# Compare training and testing accuracy
print("\n========== OVERFITTING CHECK ==========")

print(
    "Full Tree Training Accuracy:",
    round(tree_full.score(X_train_scaled, y_train), 4)
)

print(
    "Full Tree Testing Accuracy:",
    round(tree_full.score(X_test_scaled, y_test), 4)
)

print(
    "Depth-3 Training Accuracy:",
    round(tree_depth3.score(X_train_scaled, y_train), 4)
)

print(
    "Depth-3 Testing Accuracy:",
    round(tree_depth3.score(X_test_scaled, y_test), 4)
)


========== FULL DECISION TREE ==========
Accuracy: 0.6717
Precision: 0.3519
Recall: 0.3878
F1 Score: 0.3689
ROC-AUC: 0.5764

========== DECISION TREE - MAX DEPTH 3 ==========
Accuracy: 0.7778
Precision: 0.6471
Recall: 0.2245
F1 Score: 0.3333
ROC-AUC: 0.6997

========== OVERFITTING CHECK ==========
Full Tree Training Accuracy: 1.0
Full Tree Testing Accuracy: 0.6717
Depth-3 Training Accuracy: 0.822
Depth-3 Testing Accuracy: 0.7778


In [ ]:
# Random Forest Classifier

from sklearn.ensemble import RandomForestClassifier

# Create Random Forest with 100 trees
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Train model
rf_model.fit(X_train_scaled, y_train)

# Predictions
rf_pred = rf_model.predict(X_test_scaled)
rf_proba = rf_model.predict_proba(X_test_scaled)[:, 1]

# Evaluate model
evaluate_model(
    "RANDOM FOREST",
    y_test,
    rf_pred,
    rf_proba
)

# Feature importance
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print("\n========== RANDOM FOREST FEATURE IMPORTANCE ==========")
display(feature_importance.round(4))


========== RANDOM FOREST ==========
Accuracy: 0.7475
Precision: 0.4839
Recall: 0.3061
F1 Score: 0.375
ROC-AUC: 0.7077

========== RANDOM FOREST FEATURE IMPORTANCE ==========


,Feature,Importance
3,lead_days,0.2199
0,age,0.1968
1,distance_km,0.1947
2,fee_inr,0.1374
4,prior_no_shows,0.1057
5,reminder_sent,0.0457
7,booking_channel_Phone,0.0269
8,booking_channel_Walk-In,0.0197
9,visit_type_Consultation,0.0194
6,booking_channel_App,0.0184


In [ ]:
# GridSearchCV

from sklearn.model_selection import GridSearchCV

# Define the Random Forest parameter grid
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 8]
}

# Create base Random Forest
rf_base = RandomForestClassifier(random_state=42)

# Set up GridSearchCV
grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1,
    return_train_score=True
)

# Fit using training data only
grid_search.fit(X_train_scaled, y_train)

# Display best configuration
print("========== GRID SEARCH RESULTS ==========")

print("Best Parameters:", grid_search.best_params_)

print("Best Cross-Validation ROC-AUC:",
      round(grid_search.best_score_, 4))

# Display all configurations
grid_results = pd.DataFrame(grid_search.cv_results_)

grid_summary = grid_results[
    [
        "param_n_estimators",
        "param_max_depth",
        "mean_test_score",
        "std_test_score"
    ]
].sort_values(by="mean_test_score", ascending=False)

display(grid_summary.round(4))

# Save best tuned model
best_rf_model = grid_search.best_estimator_

========== GRID SEARCH RESULTS ==========
Best Parameters: {'max_depth': 3, 'n_estimators': 200}
Best Cross-Validation ROC-AUC: 0.7669


,param_n_estimators,param_max_depth,mean_test_score,std_test_score
1,200,3,0.7669,0.0245
0,100,3,0.7658,0.0211
3,200,5,0.7652,0.0174
2,100,5,0.7632,0.0156
5,200,8,0.7612,0.0274
4,100,8,0.7555,0.0315


Compare all the models

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# 1. Logistic Regression Pipeline
log_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

# 2. Depth-3 Decision Tree
tree_pipeline = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

# 3. Original Random Forest
rf_pipeline = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# 4. Tuned Random Forest
tuned_rf_pipeline = grid_search.best_estimator_


# Compare models using 5-fold cross-validation
models = {
    "Logistic Regression": log_pipeline,
    "Decision Tree Depth 3": tree_pipeline,
    "Random Forest 100 Trees": rf_pipeline,
    "Tuned Random Forest": tuned_rf_pipeline
}

cv_results = []

for name, model in models.items():

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="roc_auc",
        n_jobs=-1
    )

    cv_results.append({
        "Model": name,
        "Mean ROC-AUC": scores.mean(),
        "Std ROC-AUC": scores.std()
    })

# Display comparison
cv_comparison = pd.DataFrame(cv_results)

cv_comparison = cv_comparison.sort_values(
    by="Mean ROC-AUC",
    ascending=False
)

print("========== 5-FOLD CROSS-VALIDATION COMPARISON ==========")

display(cv_comparison.round(4))

========== 5-FOLD CROSS-VALIDATION COMPARISON ==========


,Model,Mean ROC-AUC,Std ROC-AUC
0,Logistic Regression,0.7737,0.0137
3,Tuned Random Forest,0.7673,0.0245
2,Random Forest 100 Trees,0.7466,0.0295
1,Decision Tree Depth 3,0.6784,0.0395


In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Create Logistic Regression pipeline
threshold_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

# Generate out-of-fold probabilities using training data only
oof_probabilities = cross_val_predict(
    threshold_pipeline,
    X_train,
    y_train,
    cv=5,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

# Evaluate different thresholds
threshold_results = []

for threshold in [0.50, 0.45, 0.40, 0.35, 0.30, 0.25, 0.20]:

    oof_predictions = (
        oof_probabilities >= threshold
    ).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_train, oof_predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_train, oof_predictions, zero_division=0
        ),
        "F1 Score": f1_score(
            y_train, oof_predictions, zero_division=0
        ),
        "Flagged Appointments": int(oof_predictions.sum())
    })

threshold_df = pd.DataFrame(threshold_results)

print("========== THRESHOLD COMPARISON ==========")

display(threshold_df.round(4))

# Select threshold with highest F1 score
best_threshold = threshold_df.loc[
    threshold_df["F1 Score"].idxmax(),
    "Threshold"
]

print("\nBest Threshold:", best_threshold)

========== THRESHOLD COMPARISON ==========


,Threshold,Precision,Recall,F1 Score,Flagged Appointments
0,0.50,0.5753,0.2485,0.3471,73
1,0.45,0.5758,0.3373,0.4254,99
2,0.40,0.5528,0.4024,0.4658,123
3,0.35,0.5130,0.4675,0.4892,154
4,0.30,0.4688,0.5325,0.4986,192
5,0.25,0.4232,0.6036,0.4976,241
6,0.20,0.3819,0.6982,0.4937,309



Best Threshold: 0.3


In [ ]:
# Final selected threshold
FINAL_THRESHOLD = best_threshold

# Train the final Logistic Regression model
final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

final_model.fit(X_train, y_train)

# Predict probabilities on untouched test data
final_test_probabilities = final_model.predict_proba(X_test)[:, 1]

# Apply selected threshold
final_test_predictions = (
    final_test_probabilities >= FINAL_THRESHOLD
).astype(int)

# Calculate final metrics
final_accuracy = accuracy_score(y_test, final_test_predictions)

final_precision = precision_score(
    y_test, final_test_predictions, zero_division=0
)

final_recall = recall_score(
    y_test, final_test_predictions, zero_division=0
)

final_f1 = f1_score(
    y_test, final_test_predictions, zero_division=0
)

final_auc = roc_auc_score(y_test, final_test_probabilities)

print("========== FINAL MODEL TEST RESULTS ==========")

print("Selected Model: Logistic Regression")
print("Selected Threshold:", FINAL_THRESHOLD)

print("Accuracy:", round(final_accuracy, 4))
print("Precision:", round(final_precision, 4))
print("Recall:", round(final_recall, 4))
print("F1 Score:", round(final_f1, 4))
print("ROC-AUC:", round(final_auc, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_test,
    final_test_predictions,
    labels=[0, 1]
))

print("\nClassification Report:")
print(classification_report(
    y_test,
    final_test_predictions,
    target_names=["Attended (0)", "No-Show (1)"],
    zero_division=0
))

========== FINAL MODEL TEST RESULTS ==========
Selected Model: Logistic Regression
Selected Threshold: 0.3
Accuracy: 0.7929
Precision: 0.5714
Recall: 0.6531
F1 Score: 0.6095
ROC-AUC: 0.7889

Confusion Matrix:
[[125  24]
 [ 17  32]]

Classification Report:
              precision    recall  f1-score   support

Attended (0)       0.88      0.84      0.86       149
 No-Show (1)       0.57      0.65      0.61        49

    accuracy                           0.79       198
   macro avg       0.73      0.75      0.73       198
weighted avg       0.80      0.79      0.80       198



In [ ]:
# Create test risk scores
test_risk_scores = pd.DataFrame({
    "appointment_id": appointments.loc[X_test.index, "appointment_id"].values,
    "no_show": y_test.values,
    "risk_prob": final_test_probabilities,
    "flag": final_test_predictions
})

# Sort by risk probability, highest first
test_risk_scores = test_risk_scores.sort_values(
    by="risk_prob",
    ascending=False
)

# Save CSV
OUTPUT_PATH = PROJECT_DIR / "test_risk_scores.csv"

test_risk_scores.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Saved:", OUTPUT_PATH)
print("Shape:", test_risk_scores.shape)
print("\nFlag distribution:")
print(test_risk_scores["flag"].value_counts())

print("\nTop 10 highest-risk appointments:")
display(test_risk_scores.head(10))

print("\nFile exists:", OUTPUT_PATH.exists())

Saved: /content/careloop-no-show-guard/test_risk_scores.csv
Shape: (198, 4)

Flag distribution:
flag
0    142
1     56
Name: count, dtype: int64

Top 10 highest-risk appointments:


,appointment_id,no_show,risk_prob,flag
68,A0635,1,0.937204,1
24,A0544,1,0.924066,1
141,A0293,1,0.919634,1
106,A0950,1,0.818616,1
114,A0996,0,0.766710,1
69,A0067,0,0.749805,1
172,A0769,1,0.744427,1
10,A0364,0,0.731920,1
160,A0806,1,0.703462,1
173,A0560,0,0.701898,1



File exists: True


In [ ]:
# Validate test risk scores
print("========== CSV VALIDATION ==========")

print("Shape:", test_risk_scores.shape)

print("Columns:", test_risk_scores.columns.tolist())

print("Missing values:")
print(test_risk_scores.isnull().sum())

print("Unique appointment IDs:",
      test_risk_scores["appointment_id"].nunique())

print("Valid risk probabilities:",
      test_risk_scores["risk_prob"].between(0, 1).all())

print("Valid flags:",
      test_risk_scores["flag"].isin([0, 1]).all())

print("Actual test rows match:",
      set(test_risk_scores["appointment_id"]) ==
      set(appointments.loc[X_test.index, "appointment_id"]))

print("\nCSV validation completed.")

========== CSV VALIDATION ==========
Shape: (198, 4)
Columns: ['appointment_id', 'no_show', 'risk_prob', 'flag']
Missing values:
appointment_id    0
no_show           0
risk_prob         0
flag              0
dtype: int64
Unique appointment IDs: 198
Valid risk probabilities: True
Valid flags: True
Actual test rows match: True

CSV validation completed.


In [ ]:
from google.colab import files
files.download(str(OUTPUT_PATH))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>